# Baseline experiments

Planned notebook for baseline extractive QA experiments.

In [25]:
import torch
import pandas as pd
import numpy as np
import os
import random
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from tqdm.auto import tqdm
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup



from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering
)

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch: 2.15.0
MPS available: True


In [10]:
train_path = "../data/processed/squad2/train_1_percent.parquet"
val_path = "../data/processed/squad2/validation.parquet"

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

display(train_df.head())

Training samples: 1303
Validation samples: 11873


,id,title,context,question,answers,answer_texts,answer_starts,is_impossible
0,56dec2c73277331400b4d717,Arnold_Schwarzenegger,"In its April 2010 report, Progressive ethics w...",What group awarded Schwarzenegger the title of...,{'text': ['Citizens for Responsibility and Eth...,[Citizens for Responsibility and Ethics in Was...,[60],False
1,5acf9a3777cf76001a685416,Federalism,The terms 'federalism' and 'confederalism' bot...,What did James Madison referred to the new Uni...,"{'text': [], 'answer_start': []}",[],[],True
2,57298edfaf94a219006aa505,Insect,Chemical communications in animals rely on a v...,Kairomones are a type of what?,"{'text': ['semiochemicals'], 'answer_start': [...",[semiochemicals],[391],False
3,5a5cd2285e8782001a9d5db4,Computer_security,If access is gained to a car's internal contro...,What does an external controller area network ...,"{'text': [], 'answer_start': []}",[],[],True
4,5709dce26d058f1900182be0,United_States_dollar,Early releases of the Washington coin included...,What do vending machines commonly only accept?,"{'text': ['$1 bills'], 'answer_start': [1223]}",[$1 bills],[1223],False


In [11]:
print("Columns:", train_df.columns.tolist())

for i in range(5):
    print(f"\nExample {i}")
    print("Question:", train_df.iloc[i]["question"])
    print("Context:", train_df.iloc[i]["context"][:200])
    print("Answers:", train_df.iloc[i]["answers"])
    print("Impossible:", train_df.iloc[i]["is_impossible"])

Columns: ['id', 'title', 'context', 'question', 'answers', 'answer_texts', 'answer_starts', 'is_impossible']

Example 0
Question: What group awarded Schwarzenegger the title of one of the 11 "worst governors" in a 2010 report?
Context: In its April 2010 report, Progressive ethics watchdog group Citizens for Responsibility and Ethics in Washington named Schwarzenegger one of 11 "worst governors" in the United States because of variou
Answers: {'text': array(['Citizens for Responsibility and Ethics in Washington'],
      dtype=object), 'answer_start': array([60], dtype=int32)}
Impossible: False

Example 1
Question: What did James Madison referred to the new United Nations as? 
Context: The terms 'federalism' and 'confederalism' both have a root in the Latin word foedus, meaning treaty, pact or covenant. Their common meaning until the late eighteenth century was a simple league or in
Answers: {'text': array([], dtype=object), 'answer_start': array([], dtype=int32)}
Impossible: True

Examp

In [12]:
print(train_df["is_impossible"].value_counts())
print("\nValidation distribution:")
print(val_df["is_impossible"].value_counts())

is_impossible
False    868
True     435
Name: count, dtype: int64

Validation distribution:
is_impossible
True     5945
False    5928
Name: count, dtype: int64


In [14]:

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

print("BERT QA model loaded!")

W1128 21:31:09.877000 30837 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

BERT QA model loaded!


In [17]:

MAX_LENGTH = 384
DOC_STRIDE = 128

def prepare_features(df):
    questions = df["question"].fillna("").str.strip().tolist()
    contexts = df["context"].fillna("").tolist()

    encodings = tokenizer(
        questions,
        contexts,
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = encodings.pop("overflow_to_sample_mapping")
    offsets = encodings.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for feature_idx, offset in enumerate(offsets):
        sample_idx = sample_mapping[feature_idx]
        row = df.iloc[sample_idx]

        answer = row["answers"]
        answer_texts = answer["text"]
        answer_starts = answer["answer_start"]

        sequence_ids = encodings.sequence_ids(feature_idx)

        # Find context token boundaries
        context_positions = [
            i for i, seq_id in enumerate(sequence_ids)
            if seq_id == 1
        ]

        cls_index = encodings["input_ids"][feature_idx].index(
            tokenizer.cls_token_id
        )

        # Unanswerable question
        if len(answer_texts) == 0 or bool(row["is_impossible"]):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = int(answer_starts[0])
        end_char = start_char + len(answer_texts[0])

        context_start = context_positions[0]
        context_end = context_positions[-1]

        # Check whether the complete answer is in this feature
        if (
            offset[context_start][0] > start_char
            or offset[context_end][1] < end_char
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Find answer start token
        token_start = context_start

        while (
            token_start <= context_end
            and offset[token_start][0] <= start_char
        ):
            token_start += 1

        token_start -= 1

        # Find answer end token
        token_end = context_end

        while (
            token_end >= context_start
            and offset[token_end][1] >= end_char
        ):
            token_end -= 1

        token_end += 1

        start_positions.append(token_start)
        end_positions.append(token_end)

    encodings["start_positions"] = start_positions
    encodings["end_positions"] = end_positions

    return encodings

In [18]:
train_features = prepare_features(train_df)

print("Training examples:", len(train_df))
print("Training features:", len(train_features["input_ids"]))
print("Input shape:", len(train_features["input_ids"][0]))
print("First start position:", train_features["start_positions"][0])
print("First end position:", train_features["end_positions"][0])

Training examples: 1303
Training features: 1321
Input shape: 384
First start position: 35
First end position: 41


In [ ]:

starts = np.array(train_features["start_positions"])
ends = np.array(train_features["end_positions"])

print("Total features:", len(starts))
print("CLS/no-answer features:", np.sum(starts == 0))
print("Answer-containing features:", np.sum(starts != 0))
print("Invalid spans:", np.sum(ends < starts))

Total features: 1321
CLS/no-answer features: 450
Answer-containing features: 871
Invalid spans: 0


In [21]:


device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

train_dataset = TensorDataset(
    torch.tensor(train_features["input_ids"], dtype=torch.long),
    torch.tensor(train_features["attention_mask"], dtype=torch.long),
    torch.tensor(train_features["start_positions"], dtype=torch.long),
    torch.tensor(train_features["end_positions"], dtype=torch.long)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

model.to(device)

print("Training device:", device)
print("Number of training batches:", len(train_loader))

Training device: mps
Number of training batches: 166


In [22]:
val_features = prepare_features(val_df)

print("Validation examples:", len(val_df))
print("Validation features:", len(val_features["input_ids"]))

Validation examples: 11873
Validation features: 12134


In [23]:
val_dataset = TensorDataset(
    torch.tensor(val_features["input_ids"], dtype=torch.long),
    torch.tensor(val_features["attention_mask"], dtype=torch.long),
    torch.tensor(val_features["start_positions"], dtype=torch.long),
    torch.tensor(val_features["end_positions"], dtype=torch.long)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

print("Validation batches:", len(val_loader))

Validation batches: 1517


In [26]:


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

EPOCHS = 2
LEARNING_RATE = 3e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print("Total training steps:", total_steps)
print("Device:", device)

Total training steps: 332
Device: mps


In [ ]:
best_val_loss = float("inf")

os.makedirs("../models/bert_squad2_baseline", exist_ok=True)

for epoch in range(EPOCHS):

    # Training
    model.train()
    total_train_loss = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for batch in progress:

        input_ids, attention_mask, start_positions, end_positions = [
            x.to(device) for x in batch
        ]

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            start_positions=start_positions,
            end_positions=end_positions
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation loss
    model.eval()
    total_val_loss = 0

    with torch.no_grad():

        for batch in tqdm(val_loader, desc="Validation"):

            input_ids, attention_mask, start_positions, end_positions = [
                x.to(device) for x in batch
            ]

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                start_positions=start_positions,
                end_positions=end_positions
            )

            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"\nEpoch {epoch + 1}")
    print(f"Training loss:   {avg_train_loss:.4f}")
    print(f"Validation loss: {avg_val_loss:.4f}")

    # Save best checkpoint
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        model.save_pretrained("../models/bert_squad2_baseline")
        tokenizer.save_pretrained("../models/bert_squad2_baseline")

        print("Best model checkpoint saved!")

print("\nTraining completed!")

Epoch 1/2:   0%|          | 0/166 [00:00<?, ?it/s]